# Water Quality Data Cleaning (CWB)

This notebook cleans the raw CWB beach water quality dataset
(`resources/raw_data/water_quality_cwb.csv`) and prepares it for the Kaimaemae
machine learning pipeline.

Each row in the raw data is one bacteria sample taken at one Oahu beach on one
date. The target we eventually predict is the enterococcus bacteria count, and a
beach is labeled unsafe when that count is above the EPA Beach Action Value (BAV)
of 130 CFU per 100 mL.

Cleaning goals for this dataset:

1. Drop the units descriptor row that sits under the header.
2. Convert the `time` column to a plain `YYYY-MM-DD` date.
3. Keep only Oahu sites using the lat / lon bounding box.
4. Drop rows where enterococcus is missing, since those cannot be labeled.
5. If a beach has more than one sample on the same date, keep the highest
   enterococcus reading (most conservative).
6. Keep only beaches with at least 50 historical samples so each location has
   enough data for reliable model training.
7. Add the `unsafe` binary target label and a `month` feature for the model.

Output: `resources/clean_data/clean_water_quality_data.csv`


In [1]:
## Imports and configuration

# pandas does the table work. The path constants point at the raw input and the
# clean output. The bounding box and thresholds come straight from the project
# overview so every magic number lives in one place.
import os

import pandas as pd

# Notebook lives in backend/data_scripts/ so go up two levels to the project root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
RAW_PATH = os.path.join(PROJECT_ROOT, "resources", "raw_data", "water_quality_cwb.csv")
CLEAN_DIR = os.path.join(PROJECT_ROOT, "resources", "clean_data")
OUTPUT_PATH = os.path.join(CLEAN_DIR, "clean_water_quality_data.csv")

# Oahu lat / lon bounding box (same box used in the source query URL).
LAT_MIN, LAT_MAX = 21.2, 21.7
LON_MIN, LON_MAX = -158.3, -157.6

# EPA Beach Action Value: at or below 130 is safe, above 130 is unsafe.
BAV_THRESHOLD = 130

# Minimum samples a beach must have to be kept for model training.
MIN_SAMPLES_PER_BEACH = 50

os.makedirs(CLEAN_DIR, exist_ok=True)


In [2]:
## Load the raw data and drop the units row

# The CSV header is the real column names, but the very first data row is a units
# descriptor (UTC, degrees_north, 1/(100-mL), ...) rather than a real sample.
# skiprows=[1] removes only that descriptor row while keeping the header.
raw = pd.read_csv(RAW_PATH, skiprows=[1])

print("Raw shape:", raw.shape)
raw.head()


Raw shape: (52639, 6)


,time,location_id,location_name,latitude,longitude,enterococcus
0,1973-06-05T19:10:00Z,189,Ewa Beach,21.313597,-157.99200,NaN
1,1973-06-05T20:15:00Z,188,Kahe Point Park,21.356148,-158.13020,NaN
2,1973-06-05T20:40:00Z,183,Waianae Beach Shoreline,21.441620,-158.19101,NaN
3,1973-06-05T22:05:00Z,184,MAKUA,21.545834,-158.24028,NaN
4,1973-06-12T18:19:00Z,349,AHUA POINT,21.322500,-157.91583,NaN


In [3]:
## Convert time to a plain date

# The raw time column is a full UTC timestamp. The model only cares about the
# calendar date, which is also the key used later to join rainfall features.
# We parse the timestamp and keep just the date portion as YYYY-MM-DD text.
df = raw.copy()
df["date"] = pd.to_datetime(df["time"], utc=True).dt.strftime("%Y-%m-%d")
df = df.drop(columns=["time"])

print("Date range:", df["date"].min(), "to", df["date"].max())
df.head()


Date range: 1973-06-05 to 2024-06-13


,location_id,location_name,latitude,longitude,enterococcus,date
0,189,Ewa Beach,21.313597,-157.99200,NaN,1973-06-05
1,188,Kahe Point Park,21.356148,-158.13020,NaN,1973-06-05
2,183,Waianae Beach Shoreline,21.441620,-158.19101,NaN,1973-06-05
3,184,MAKUA,21.545834,-158.24028,NaN,1973-06-05
4,349,AHUA POINT,21.322500,-157.91583,NaN,1973-06-12


In [4]:
## Filter to Oahu sites only

# The source query was already limited to Oahu, but we re-apply the lat / lon
# bounding box here so the cleaning step is self contained and safe even if the
# raw file is ever re-exported with a wider extent.
before = len(df)
df = df[
    df["latitude"].between(LAT_MIN, LAT_MAX)
    & df["longitude"].between(LON_MIN, LON_MAX)
]
print(f"Oahu filter: kept {len(df)} of {before} rows")
df.head()


Oahu filter: kept 52639 of 52639 rows


,location_id,location_name,latitude,longitude,enterococcus,date
0,189,Ewa Beach,21.313597,-157.99200,NaN,1973-06-05
1,188,Kahe Point Park,21.356148,-158.13020,NaN,1973-06-05
2,183,Waianae Beach Shoreline,21.441620,-158.19101,NaN,1973-06-05
3,184,MAKUA,21.545834,-158.24028,NaN,1973-06-05
4,349,AHUA POINT,21.322500,-157.91583,NaN,1973-06-12


In [5]:
## Drop rows with missing enterococcus

# Enterococcus is both the model target and the basis for the safe / unsafe
# label. A row with no reading cannot be labeled, so we coerce the column to
# numeric (turning any non-numeric text into NaN) and drop those rows.
df["enterococcus"] = pd.to_numeric(df["enterococcus"], errors="coerce")

before = len(df)
df = df.dropna(subset=["enterococcus"])
print(f"Dropped missing enterococcus: kept {len(df)} of {before} rows")
df.head()


Dropped missing enterococcus: kept 48195 of 52639 rows


,location_id,location_name,latitude,longitude,enterococcus,date
1633,162,"Public Bath Beach, Waikiki",21.267614,-157.82265,4.0,1986-07-23
1634,152,ALA MOANA PARK EWA END,21.294445,-157.85777,1.0,1986-07-28
1635,157,KAHANAMOKU LAGOON-DIAMOND HEAD,21.285833,-157.84111,7.0,1986-07-28
1636,159,"Gray's Beach, Waikiki",21.277185,-157.83138,4.0,1986-07-28
1638,152,ALA MOANA PARK EWA END,21.294445,-157.85777,1.0,1986-08-04


In [6]:
## Collapse same day duplicate samples

# A beach can be sampled more than once on the same date. For a safety model the
# most conservative reading is the right one to keep, so for each beach and date
# we sort by enterococcus and keep the single highest value.
before = len(df)
df = (
    df.sort_values("enterococcus", ascending=False)
    .drop_duplicates(subset=["location_id", "date"], keep="first")
    .reset_index(drop=True)
)
print(f"Collapsed duplicates: kept {len(df)} of {before} rows")
df.head()


Collapsed duplicates: kept 47295 of 48195 rows


,location_id,location_name,latitude,longitude,enterococcus,date
0,342,KEEHI LAGOON POINT X,21.334167,-157.89640,84000.0,1996-03-04
1,302,KAELEPULU STREAM,21.399445,-157.72972,70000.0,1995-02-28
2,320,ALA MOANA BRIDGE,21.290833,-157.84334,50000.0,1991-11-30
3,320,ALA MOANA BRIDGE,21.290833,-157.84334,46000.0,1996-11-13
4,321,MCCULLY STREET BRIDGE,21.291390,-157.83556,45000.0,1990-10-17


In [7]:
## Keep only beaches with enough history

# A beach needs a reasonable number of samples for the model to learn its
# behavior. We count samples per location and drop any beach below the
# MIN_SAMPLES_PER_BEACH threshold.
counts = df["location_id"].value_counts()
keep_ids = counts[counts >= MIN_SAMPLES_PER_BEACH].index

before_rows = len(df)
before_beaches = df["location_id"].nunique()
df = df[df["location_id"].isin(keep_ids)].reset_index(drop=True)

print(f"Beaches: kept {df['location_id'].nunique()} of {before_beaches}")
print(f"Rows: kept {len(df)} of {before_rows}")
df.head()


Beaches: kept 107 of 203
Rows: kept 45553 of 47295


,location_id,location_name,latitude,longitude,enterococcus,date
0,342,KEEHI LAGOON POINT X,21.334167,-157.89640,84000.0,1996-03-04
1,302,KAELEPULU STREAM,21.399445,-157.72972,70000.0,1995-02-28
2,320,ALA MOANA BRIDGE,21.290833,-157.84334,50000.0,1991-11-30
3,320,ALA MOANA BRIDGE,21.290833,-157.84334,46000.0,1996-11-13
4,321,MCCULLY STREET BRIDGE,21.291390,-157.83556,45000.0,1990-10-17


In [8]:
## Add the model label and month feature

# unsafe is the binary classification target: 1 when the reading is above the
# 130 CFU BAV threshold, otherwise 0. month captures the wet season versus dry
# season signal and is derived straight from the date.
df["unsafe"] = (df["enterococcus"] > BAV_THRESHOLD).astype(int)
df["month"] = pd.to_datetime(df["date"]).dt.month

# Order the columns to match the project master schema for this dataset.
df = df[
    [
        "date",
        "location_id",
        "location_name",
        "latitude",
        "longitude",
        "enterococcus",
        "unsafe",
        "month",
    ]
]

print("Unsafe rate:", round(df["unsafe"].mean(), 4))
df.head()


Unsafe rate: 0.0333


,date,location_id,location_name,latitude,longitude,enterococcus,unsafe,month
0,1996-03-04,342,KEEHI LAGOON POINT X,21.334167,-157.89640,84000.0,1,3
1,1995-02-28,302,KAELEPULU STREAM,21.399445,-157.72972,70000.0,1,2
2,1991-11-30,320,ALA MOANA BRIDGE,21.290833,-157.84334,50000.0,1,11
3,1996-11-13,320,ALA MOANA BRIDGE,21.290833,-157.84334,46000.0,1,11
4,1990-10-17,321,MCCULLY STREET BRIDGE,21.291390,-157.83556,45000.0,1,10


In [9]:
## Export the cleaned dataset

# Write the cleaned, labeled table to the clean_data folder. This file is the
# input to the rainfall merge and feature engineering step that follows.
df.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(df):,} rows to {OUTPUT_PATH}")
print(f"Beaches: {df['location_id'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")


Wrote 45,553 rows to c:\Users\ralph\projects\kaimaemae\resources\clean_data\clean_water_quality_data.csv
Beaches: 107
Date range: 1986-07-23 to 2024-06-13
